In [2]:
import numpy as np

from scipy.stats import skew
from scipy.stats import kurtosis

from scipy.signal import welch

In [3]:
windows = np.load(
    "../processed/S001_R04_windows.npy"
)

labels = np.load(
    "../processed/S001_R04_labels.npy"
)

print(windows.shape)

(15, 7, 27, 80)


In [4]:
def extract_11_features(signal, fs=160):

    features = []

    # 1 Mean
    features.append(np.mean(signal))

    # 2 Variance
    features.append(np.var(signal))

    # 3 Skewness
    features.append(skew(signal))

    # 4 Kurtosis
    features.append(kurtosis(signal))

    # 5 Zero Crossing Count
    zc = np.sum(
        np.diff(
            np.sign(signal)
        ) != 0
    )

    features.append(zc)

    # 6 Area
    features.append(
        np.trapezoid(np.abs(signal))
    )

    # 7 Range
    features.append(
        np.max(signal) - np.min(signal)
    )

    # PSD
    freqs, psd = welch(
        signal,
        fs=fs
    )

    # 8 Delta
    delta = np.sum(
        psd[
            (freqs >= 0.5) &
            (freqs < 4)
        ]
    )

    # 9 Theta
    theta = np.sum(
        psd[
            (freqs >= 4) &
            (freqs < 8)
        ]
    )

    # 10 Alpha
    alpha = np.sum(
        psd[
            (freqs >= 8) &
            (freqs < 12)
        ]
    )

    # 11 Beta
    beta = np.sum(
        psd[
            (freqs >= 12) &
            (freqs < 30)
        ]
    )

    features.extend([
        delta,
        theta,
        alpha,
        beta
    ])

    return np.array(features)

In [5]:
def extract_11_features_v2(signal, fs=160):

    features = []

    # 1 Mean
    features.append(np.mean(signal))

    # 2 Variance
    features.append(np.var(signal))

    # 3 Skewness
    features.append(skew(signal))

    # 4 Kurtosis
    features.append(kurtosis(signal))

    # 5 Zero Crossing Count
    zc = np.sum(
        np.diff(
            np.sign(signal)
        ) != 0
    )

    features.append(zc)

    # 6 Absolute Area Under Signal
    features.append(
        np.trapezoid(np.abs(signal))
    )

    # 7 Peak-to-Peak (Range)
    features.append(
        np.max(signal) - np.min(signal)
    )

    # Power Spectral Density
    freqs, psd = welch(
        signal,
        fs=fs,
        nperseg=len(signal)
    )

    # Total Power
    total_power = np.trapezoid(
        psd,
        freqs
    )

    # Safety check
    if total_power == 0:
        total_power = 1e-10

    # Delta (0.5-4 Hz)
    delta_mask = (
        (freqs >= 0.5) &
        (freqs < 4)
    )

    delta_power = np.trapezoid(
        psd[delta_mask],
        freqs[delta_mask]
    )

    delta = delta_power / total_power

    # Theta (4-8 Hz)
    theta_mask = (
        (freqs >= 4) &
        (freqs < 8)
    )

    theta_power = np.trapezoid(
        psd[theta_mask],
        freqs[theta_mask]
    )

    theta = theta_power / total_power

    # Alpha (8-12 Hz)
    alpha_mask = (
        (freqs >= 8) &
        (freqs < 12)
    )

    alpha_power = np.trapezoid(
        psd[alpha_mask],
        freqs[alpha_mask]
    )

    alpha = alpha_power / total_power

    # Beta (12-30 Hz)
    beta_mask = (
        (freqs >= 12) &
        (freqs < 30)
    )

    beta_power = np.trapezoid(
        psd[beta_mask],
        freqs[beta_mask]
    )

    beta = beta_power / total_power

    features.extend([
        delta,
        theta,
        alpha,
        beta
    ])

    return np.array(features)

In [6]:
test_signal = windows[0,0,0]

feat = extract_11_features(
    test_signal
)

print(feat)

print(feat.shape)

[5.59749780e-01 3.85595419e-03 4.93488080e-01 6.65768392e-01
 0.00000000e+00 4.41895695e+01 3.28894948e-01 5.52487127e-04
 8.29148666e-04 3.65920452e-04 7.55212771e-04]
(11,)


C:\Users\gurle\AppData\Local\Temp\ipykernel_13792\3522270156.py:37: UserWarning: nperseg=256 is greater than signal length max(len(x), len(y)) = 80, using nperseg = 80
  freqs, psd = welch(


In [7]:
test_signal = windows[0,0,0]

old_feat = extract_11_features(test_signal)
new_feat = extract_11_features_v2(test_signal)

print("Old PSD Features:")
print(old_feat[7:])

print()

print("New PSD Features:")
print(new_feat[7:])

print()

print("Sum of Relative Powers:")
print(np.sum(new_feat[7:]))

Old PSD Features:
[0.00055249 0.00082915 0.00036592 0.00075521]

New PSD Features:
[0.         0.15067236 0.06649483 0.23275826]

Sum of Relative Powers:
0.4499254439650442


C:\Users\gurle\AppData\Local\Temp\ipykernel_13792\3522270156.py:37: UserWarning: nperseg=256 is greater than signal length max(len(x), len(y)) = 80, using nperseg = 80
  freqs, psd = welch(


In [8]:
all_features = []

for trial in windows:

    trial_features = []

    for window in trial:

        window_features = []

        for channel in window:

            feats = extract_11_features(
                channel
            )

            window_features.extend(
                feats
            )

        trial_features.append(
            window_features
        )

    all_features.append(
        trial_features
    )

all_features = np.array(
    all_features
)

C:\Users\gurle\AppData\Local\Temp\ipykernel_13792\3522270156.py:37: UserWarning: nperseg=256 is greater than signal length max(len(x), len(y)) = 80, using nperseg = 80
  freqs, psd = welch(


In [9]:
print(all_features.shape)

(15, 7, 297)


In [10]:
print(all_features[0].shape)

print(all_features[0,0].shape)

(7, 297)
(297,)


In [11]:
np.save(
    "../processed/S001_R04_features.npy",
    all_features
)

np.save(
    "../processed/S001_R04_labels.npy",
    labels
)

print("Saved successfully")

Saved successfully


In [12]:
for i in range(5):

    sig = windows[0, i, 0]

    feat1 = extract_11_features(sig)
    feat2 = extract_11_features_v2(sig)

    print(f"\nWindow {i+1}")

    print("Old:")
    print(feat1[7:])

    print("New:")
    print(feat2[7:])

    print("Sum:")
    print(np.sum(feat2[7:]))


Window 1
Old:
[0.00055249 0.00082915 0.00036592 0.00075521]
New:
[0.         0.15067236 0.06649483 0.23275826]
Sum:
0.4499254439650442

Window 2
Old:
[0.00020344 0.00010395 0.00013789 0.0005984 ]
New:
[0.         0.03794822 0.05033932 0.36177403]
Sum:
0.45006156687386656

Window 3
Old:
[1.81076510e-04 8.13989803e-05 1.75846394e-04 4.23572201e-04]
New:
[0.         0.03802053 0.08213583 0.32721357]
Sum:
0.44736992492709343

Window 4
Old:
[0.00057875 0.00049394 0.00010207 0.00069989]
New:
[0.         0.11938904 0.0246712  0.28842507]
Sum:
0.43248531624801295

Window 5
Old:
[4.18015393e-05 2.71309120e-04 2.14834769e-04 5.16189312e-04]
New:
[0.         0.10497672 0.08312529 0.32619968]
Sum:
0.514301686771702


C:\Users\gurle\AppData\Local\Temp\ipykernel_13792\3522270156.py:37: UserWarning: nperseg=256 is greater than signal length max(len(x), len(y)) = 80, using nperseg = 80
  freqs, psd = welch(


In [13]:
test_signal = windows[0,0,0]

freqs, psd = welch(
    test_signal,
    fs=160,
    nperseg=len(test_signal)
)

print(freqs)

[ 0.  2.  4.  6.  8. 10. 12. 14. 16. 18. 20. 22. 24. 26. 28. 30. 32. 34.
 36. 38. 40. 42. 44. 46. 48. 50. 52. 54. 56. 58. 60. 62. 64. 66. 68. 70.
 72. 74. 76. 78. 80.]


In [14]:
X = np.load("../processed/X_no_norm.npy")

print(X.shape)

(4635, 7, 297)


In [15]:
sample = X[0,0]

print(sample.shape)

(297,)


In [16]:
sample = sample.reshape(27,11)

print(sample)

[[-5.52267947e-06  5.09748287e-10  4.93488080e-01  6.65768392e-01
   1.30000000e+01  1.45872735e-03  1.19582921e-04  7.30375291e-11
   1.09611549e-10  4.83738433e-11  9.98373936e-11]
 [-1.69196582e-06  1.98793263e-10  5.96278007e-02 -3.12045409e-01
   1.70000000e+01  8.98539888e-04  6.49745664e-05  7.01637000e-12
   1.04223693e-11  2.06383593e-11  7.40281222e-11]
 [ 2.06049461e-06  6.56944502e-11 -9.21811951e-02 -2.75999965e-01
   1.40000000e+01  5.45979857e-04  3.91541038e-05  3.24834479e-12
   7.64130124e-12  9.45883815e-12  1.92999201e-11]
 [ 1.02405502e-06  6.37659038e-10  1.78245211e-01 -6.20992510e-01
   1.70000000e+01  1.66289411e-03  1.19217521e-04  3.08847891e-12
   2.72891017e-11  5.05373628e-11  1.02696982e-10]
 [-1.05813974e-06  2.33157690e-10  5.25275923e-02 -8.44830649e-01
   1.30000000e+01  1.04240219e-03  6.30389835e-05  8.85148027e-12
   2.00949435e-11  2.68382167e-11  3.77969987e-11]
 [-6.17763739e-06  1.77785164e-10  9.89331875e-03 -1.76331711e-01
   1.20000000e+01  

In [17]:
print(sample[:,7:11])

[[7.30375291e-11 1.09611549e-10 4.83738433e-11 9.98373936e-11]
 [7.01637000e-12 1.04223693e-11 2.06383593e-11 7.40281222e-11]
 [3.24834479e-12 7.64130124e-12 9.45883815e-12 1.92999201e-11]
 [3.08847891e-12 2.72891017e-11 5.05373628e-11 1.02696982e-10]
 [8.85148027e-12 2.00949435e-11 2.68382167e-11 3.77969987e-11]
 [1.95507248e-11 9.78097503e-12 3.40169345e-12 8.85773889e-12]
 [2.52569604e-11 5.47625080e-11 3.45295195e-11 4.88914207e-11]
 [1.34369317e-11 3.71591715e-11 2.18453805e-11 2.85196538e-11]
 [1.19808767e-12 9.96352309e-12 9.09501992e-12 8.92175991e-12]
 [9.36165388e-12 4.26899881e-12 5.66970873e-12 3.63745997e-11]
 [1.54435492e-10 1.46280611e-10 4.92816541e-11 1.37466088e-10]
 [2.99830469e-11 5.46591912e-11 1.32782953e-11 1.03371328e-10]
 [4.31873502e-10 4.98091533e-10 8.87955169e-11 1.85912289e-10]
 [3.72292804e-11 6.43373281e-11 1.77634471e-11 1.23093092e-10]
 [3.07140986e-12 3.34583458e-12 1.06191843e-12 1.61276285e-12]
 [1.75674225e-12 3.17993120e-12 1.84981516e-12 2.041449

In [18]:
sample = X[0,0].reshape(27,11)

print("Mean")
print(sample[:,0])

print("Variance")
print(sample[:,1])

print("Area")
print(sample[:,5])

print("Range")
print(sample[:,6])

print("Delta")
print(sample[:,7])

print("Theta")
print(sample[:,8])

print("Alpha")
print(sample[:,9])

print("Beta")
print(sample[:,10])

Mean
[-5.52267947e-06 -1.69196582e-06  2.06049461e-06  1.02405502e-06
 -1.05813974e-06 -6.17763739e-06 -2.86201008e-06 -1.66673587e-06
  1.20358309e-06 -5.42693415e-06 -4.72068661e-08 -4.70625401e-06
 -6.06766817e-06  3.27642083e-07 -5.27601898e-06  3.94309497e-07
 -2.44095423e-06 -4.38967329e-06 -5.65259009e-06 -4.82870141e-06
 -3.71767112e-06 -8.21668413e-07 -7.43608974e-07  3.10756950e-07
 -1.18722097e-06 -3.91237236e-06  2.47089657e-06]
Variance
[5.09748287e-10 1.98793263e-10 6.56944502e-11 6.37659038e-10
 2.33157690e-10 1.77785164e-10 4.56549541e-10 2.51628142e-10
 8.27286723e-11 1.67951088e-10 7.55831551e-10 3.11604576e-10
 1.83854836e-09 3.81652449e-10 2.11977342e-11 5.84188981e-11
 1.05413310e-09 1.62294443e-09 9.29390195e-10 7.92717681e-10
 5.89356454e-10 4.45518136e-10 3.28014800e-10 1.36262768e-10
 6.51515365e-10 5.07647215e-10 6.50911418e-10]
Area
[0.00145873 0.00089854 0.00054598 0.00166289 0.0010424  0.00092666
 0.0013843  0.00105706 0.00059878 0.0008942  0.00173663 0.001

In [20]:
import mne
raw = mne.io.read_raw_edf(
    "../data/eegmmidb/files/S001/S001R04.edf",
    preload=True,
    verbose=False
)

print(raw.get_data().min())
print(raw.get_data().max())

-0.000376
0.0005949999999999999
